# Numerai 13F-HR Data Updater

このノートブックは、SEC EDGAR APIからNumerai GP LLC (CIK: 0001668527) の最新の13F-HR提出データを取得し、既存の `History_Numerai_13F-HR.xlsx` に追記します。

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
from datetime import datetime
import time

# --- 設定 ---
CIK = "0001668527"
USER_AGENT = "Kei Sanada sdk7777@gmail.com"
EXCEL_FILE = "History_Numerai_13F-HR.xlsx"
SEC_HEADERS = {"User-Agent": USER_AGENT}

### 1. 最新の13F-HR提出物を特定する

In [ ]:
def get_latest_13f_submission(cik):
    # SECの最新提出物メタデータを取得
    url = f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json"
    response = requests.get(url, headers=SEC_HEADERS)
    response.raise_for_status()
    data = response.json()
    
    recent_filings = data['filings']['recent']
    # フォームリストから最新の 13F-HR を検索
    for i, form in enumerate(recent_filings['form']):
        if form == '13F-HR':
            acc_num = recent_filings['accessionNumber'][i]
            report_date = recent_filings['reportDate'][i]
            # アクセッション番号からハイフンを除去してディレクトリ形式に
            return acc_num.replace('-', ''), report_date
    return None, None

acc_num_clean, latest_report_date = get_latest_13f_submission(CIK)
if acc_num_clean:
    print(f"最新の提出物を発見: Accession No: {acc_num_clean}, 報告基準日: {latest_report_date}")
else:
    print("13F-HR フォームが見つかりませんでした。")

### 2. XMLデータを取得してパースする

In [ ]:
def fetch_13f_data(cik, acc_num_clean, report_date):
    # informationtable.xml は通常このパスに配置される
    xml_filename = "informationtable.xml"
    xml_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num_clean}/{xml_filename}"
    
    print(f"XMLを取得中: {xml_url}")
    response = requests.get(xml_url, headers=SEC_HEADERS)
    
    # 大文字小文字の差異に対応するためリトライ
    if response.status_code != 200:
        xml_url = xml_url.replace("informationtable.xml", "InformationTable.xml")
        response = requests.get(xml_url, headers=SEC_HEADERS)
    
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, 'xml')
    info_table = []
    
    # SEC 13F のXMLスキーマ（名前空間）に対応した抽出
    # 複数のタグ名候補をチェック
    for info in soup.find_all(['infoTable', 'ns1:infoTable']):
        row = {
            'ISSUER NAME': info.find(['nameOfIssuer', 'ns1:nameOfIssuer']).text,
            'TITLE OF CLASS': info.find(['titleOfClass', 'ns1:titleOfClass']).text,
            'CUSIP': info.find(['cusip', 'ns1:cusip']).text,
            'VALUE (x$1000)': int(info.find(['value', 'ns1:value']).text),
            'SHRS OR PRN AMT': int(info.find(['sshPrnamt', 'ns1:sshPrnamt']).text),
            'SH/ PRN': info.find(['sshPrnamtType', 'ns1:sshPrnamtType']).text,
            'PUT/ CALL': info.find(['putCall', 'ns1:putCall']).text if info.find(['putCall', 'ns1:putCall']) else '',
            'INVESTMENT DISCRETION': info.find(['investmentDiscretion', 'ns1:investmentDiscretion']).text,
            'OTHER MANAGER': info.find(['otherManager', 'ns1:otherManager']).text if info.find(['otherManager', 'ns1:otherManager']) else '',
            'SOLE': int(info.find(['sole', 'ns1:sole']).text) if info.find(['sole', 'ns1:sole']) else 0,
            'SHARED': int(info.find(['shared', 'ns1:shared']).text) if info.find(['shared', 'ns1:shared']) else 0,
            'NONE': int(info.find(['none', 'ns1:none']).text) if info.find(['none', 'ns1:none']) else 0,
            'Report Date': report_date
        }
        info_table.append(row)
    
    return pd.DataFrame(info_table)

if acc_num_clean:
    new_df = fetch_13f_data(CIK, acc_num_clean, latest_report_date)
    print(f"{len(new_df)} 件のデータを取得しました。")

### 3. 既存のExcelを更新する

In [ ]:
if 'new_df' in locals() and not new_df.empty:
    if os.path.exists(EXCEL_FILE):
        existing_df = pd.read_excel(EXCEL_FILE)
        # 日付の重複チェック（文字列として比較）
        if latest_report_date in existing_df['Report Date'].astype(str).unique():
            print(f"警告: 報告日 {latest_report_date} のデータは既にExcel内に存在します。重複を避けるため追記しません。")
        else:
            # 既存データに結合
            updated_df = pd.concat([existing_df, new_df], ignore_index=True)
            updated_df.to_excel(EXCEL_FILE, index=False)
            print(f"成功: {EXCEL_FILE} に {len(new_df)} 件のデータを追記しました。")
    else:
        # 新規作成
        new_df.to_excel(EXCEL_FILE, index=False)
        print(f"成功: {EXCEL_FILE} を新規作成し {len(new_df)} 件のデータを保存しました。")
else:
    print("エラー: 取得されたデータがありません。")